# Chapter 6 — Main Comparison / Bakeoff (per-dataset Colab launcher)

baseline vs COSGD vs BOGrad vs dropout vs GradDrop, across 4 base optimisers,
**tuned on val, reported on test**, multi-seed with paired data order. Produces
the canonical record set + the **FOGO-style results table** (memory, wall-clock,
step time, accuracy, epochs/steps-to-target, convergence speed-up).

## How to run it concurrently (the time-saver)
1. Open **one Colab session per dataset** (duplicate this notebook, or just change
   `DATASET` in each).
2. Set `DATASET` in the config cell, Runtime -> GPU, **Run all**.
3. Every session writes into the **same campaign `main`** on your Drive, each in
   its own dataset subfolder, so there is no collision and they aggregate
   automatically. Everything is **resumable** (a re-run skips finished cells and
   reuses cached tuning), so a disconnect costs nothing.
4. When all datasets are done, run the **views** cell in any one session to build
   every figure + table from the combined record set.

The Ch4/Ch5 decisions are already baked into the harness: only the learning rate
(+ the minimal per-method knob — BOGrad K bounded by the per-base optimum, COSGD
norm cap) is tuned; the structural configs (sequential-negative update-stage
BOGrad; classical-GS desc-sum COSGD) are fixed.

## 1. Setup — clone, deps, mount Drive, persist results

In [ ]:
import os, subprocess, sys, importlib, pathlib, shutil
REPO_URL = "https://github.com/rayden96/MastersDissertationExperiments.git"
BRANCH   = "m0-infrastructure"
REPO_DIR = "/content/MastersDissertationExperiments"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for pkg, pn in [("sklearn", "scikit-learn"), ("datasets", "datasets")]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pn], check=True)

from google.colab import drive
drive.mount("/content/drive")
RESULTS_ROOT = "/content/drive/MyDrive/dissertation/results"
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.environ["DISSERTATION_RESULTS_ROOT"] = RESULTS_ROOT

# Persist the bakeoff output dirs on Drive (symlink local -> Drive) so concurrent
# sessions share one record set and survive disconnects.
mc = pathlib.Path(REPO_DIR) / "PaperReadyExperiments" / "30_main_comparison"
for sub in ("_core", "_sensitivity", "_scale", "_gradstats"):
    local = mc / sub
    drive_dir = pathlib.Path(RESULTS_ROOT) / "30_main_comparison" / sub
    drive_dir.mkdir(parents=True, exist_ok=True)
    if local.exists() and not local.is_symlink():
        for it in local.glob("*"):
            shutil.move(str(it), str(drive_dir / it.name))
        shutil.rmtree(local, ignore_errors=True)
    if not local.exists():
        os.symlink(drive_dir, local, target_is_directory=True)

import torch
print("cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Config — set the dataset for THIS session
One dataset per session. Default per-dataset epochs (override with `EPOCHS`):
`mnist` 10 | `emnist_balanced` 15 | `cifar10` 30 | `covertype` 20 |
`yahoo_answers` 15 | `cifar100` 50.

`MEASURE=False` (default) runs fast and gives clean memory / step-time / accuracy
curves for the FOGO table and speed-up. Set `MEASURE=True` only if you also want
the interference metrics logged per cell (much slower — the meter adds per-class
passes); the separability evidence already lives in the Ch4/Ch5 ablations.

In [ ]:
DATASET  = "cifar10"   # one of: mnist, cifar10, cifar100, emnist_balanced, covertype, yahoo_answers
SEEDS    = [2026, 2027, 2028, 2029, 2030]
EPOCHS   = None        # None = per-dataset default; or an int to override
MEASURE  = False       # True also logs interference metrics (slow)
CAMPAIGN = "main"      # keep identical across sessions so they aggregate
# CIFAR-100 ResNet is heaviest; COSGD at 100 classes is slow (expected) — consider 3 seeds there.
print("session:", DATASET, "| seeds", SEEDS, "| epochs", EPOCHS or "default",
      "| measure", MEASURE, "| campaign", CAMPAIGN)

## 3. Smoke test (optional, ~3 min) — prove the pipeline first

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python run.py --smoke
!python views.py --campaign _core/results/smoke --only 10 00

## 4. Run the bakeoff for THIS dataset
Resumable: re-run to continue. Tuning is cached under the dataset's `tune/`.

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
SEED_STR = " ".join(str(s) for s in SEEDS)
RUN_CMD = f"python run.py --datasets {DATASET} --seeds {SEED_STR} --campaign {CAMPAIGN}"
if EPOCHS:
    RUN_CMD += f" --epochs {EPOCHS}"
if not MEASURE:
    RUN_CMD += " --no-measure"
print(RUN_CMD)
!{RUN_CMD}

## 5. Build all figures + tables (run once, after ALL datasets are done)
Reads the combined `main` campaign and writes, into the campaign dir on Drive:

- `30_10_fogo_table.md` / `.json` + `30_10_fogo_table_<dataset>.tex` — **the FOGO-style table**
- `30_00_speedup.*` — convergence speed-up (the headline)
- `30_01_trajectories_<dataset>.png`, `30_02_budget_tables.md`, `30_03_final_bars_*.png`, `30_09_pareto_*.png`

then zips the small artifacts for download.

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python views.py --campaign _core/results/{CAMPAIGN}

import os, glob, zipfile
camp = f"_core/results/{CAMPAIGN}"
outzip = "/content/bakeoff_figures_tables.zip"
with zipfile.ZipFile(outzip, "w") as z:
    for pat in ("30_*.md", "30_*.tex", "30_*.json", "30_*.png"):
        for f in glob.glob(os.path.join(camp, "**", pat), recursive=True):
            z.write(f, os.path.relpath(f, camp))
print("zipped ->", outzip, "| also on Drive under", camp)
try:
    from google.colab import files; files.download(outzip)
except Exception as e:
    print("(grab it from the Files pane / Drive)", e)

## 6. Focused sweeps (optional — run once, not per dataset)
The LR / K / batch-size sensitivity (30.04-30.06) and model-scale / grad-stats
(30.07-30.08) axes. Cheap relative to the full bakeoff.

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python sensitivity.py --kind lr    --datasets cifar10 cifar100
!python sensitivity.py --kind K     --datasets cifar10 mnist emnist_balanced
!python sensitivity.py --kind batch --datasets cifar10 emnist_balanced
!python scale_gradstats.py --kind scale
!python scale_gradstats.py --kind gradstats